# Bronze — Idempotent Batch Ingestion

Loads the 8 Olist CSV files from the landing volume into `globalmart.bronze.*` Delta tables.

**Design:** file fingerprint = `path + size + modificationTime`. If a fingerprint is already logged as `INGESTED`, the file is skipped. Each row is enriched with `_source_file_name`, `_source_path`, `_ingestion_run_id`, `_ingested_at`.

**Run this notebook twice** to prove idempotency — the second run should show `SKIPPED` and zero new rows.

In [ ]:
import sys

dbutils.widgets.text(
    "repo_root",
    "",
    "Path to this repo in Databricks Repos",
)
REPO_ROOT = dbutils.widgets.get("repo_root").strip()
if not REPO_ROOT:
    raise ValueError(
        "Set the repo_root widget to your Databricks Repos path, "
        "e.g. /Repos/<username>/ecommerce-analytics"
    )
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.ingestion.idempotent_loader import (
    IngestionConfig,
    ingest_landing_zone,
    build_ingestion_summary,
)

config = IngestionConfig(
    landing_path="/Volumes/globalmart/bronze/raw_landing"
)
print("Landing path:", config.landing_path)

In [ ]:
# Verify uploaded files
landing = config.landing_path
files = dbutils.fs.ls(landing)
display(spark.createDataFrame([(f.name, f.size) for f in files], ["file", "bytes"]))

In [ ]:
# First ingestion run
run_id, results = ingest_landing_zone(spark, dbutils, config)
print("Run ID:", run_id)
display(spark.createDataFrame([r.__dict__ for r in results]))

In [ ]:
# Summary report
from pyspark.sql import functions as F

summary = build_ingestion_summary(spark, config)
display(summary.orderBy(F.col("record_count").desc()))

top_records = summary.orderBy(F.col("record_count").desc()).first()
top_columns = summary.orderBy(F.col("column_count").desc()).first()
print(f"Most records: {top_records['bronze_table']} ({top_records['record_count']:,})")
print(f"Most columns:  {top_columns['bronze_table']} ({top_columns['column_count']})")

In [ ]:
# Idempotency proof — run ingestion again
run_id_2, results_2 = ingest_landing_zone(spark, dbutils, config)
display(spark.createDataFrame([r.__dict__ for r in results_2]))

skipped = sum(1 for r in results_2 if r.status == "SKIPPED")
ingested = sum(r.records_ingested for r in results_2)
print(f"Second run: {skipped} files skipped, {ingested} new records ingested")

In [ ]:
# Ingestion audit log
from pyspark.sql import functions as F

display(
    spark.table(config.metadata_table)
    .orderBy(F.col("ingested_at").desc())
)

In [ ]:
# Export run report → save in repo + print for copy-paste
import json
from src.ingestion.idempotent_loader import build_run_report, write_run_report

report = build_run_report(
    spark,
    config=config,
    first_run_id=run_id,
    first_results=results,
    second_run_id=run_id_2,
    second_results=results_2,
)

report_path = f"{REPO_ROOT}/reports/ingestion_latest.json"
saved = write_run_report(report, report_path, dbutils=dbutils)
print(f"Saved: {saved}")
print("\n--- Copy below to share results ---")
print(json.dumps(report, indent=2))